# dataclass-training-args — faded example 1: Derive total training steps in __post_init__

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclass-training-args`. Running the beacon reports progress on the `Config: @dataclass training args` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: @dataclass training args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclass-training-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclass-training-args"
DD_SUBTOPIC = "Config: @dataclass training args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`__post_init__` runs after the auto-generated `__init__`, so every declared field is already set on `self`. It is the canonical place to compute a derived field from the raw inputs — here, total optimizer steps from epochs, dataset size, and batch size.

## Faded exercise 1

### Faded — derive total steps in `__post_init__`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> ```

Complete the `ScheduleArgs` dataclass below. It has fields `epochs`, `dataset_size`, and `batch_size` with defaults, plus a derived field `total_steps` that defaults to `0`. In `__post_init__`, set `self.total_steps` to the number of optimizer steps across all epochs, using a CEIL division so a partial final batch still counts as a step. Everything else is filled in for you.

**Fill in:** the computation of total optimizer steps as epochs times the ceil-division of dataset_size by batch_size.

In [ ]:
from dataclasses import dataclass

@dataclass
class ScheduleArgs:
    epochs: int = 3
    dataset_size: int = 1000
    batch_size: int = 32
    total_steps: int = 0

    def __post_init__(self):
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')
        self.total_steps = None  # TODO: total optimizer steps = epochs * ceil(dataset_size / batch_size)

args = ScheduleArgs(epochs=3, dataset_size=1000, batch_size=32)
print(args.total_steps)


def _test():
    import math
    a = ScheduleArgs(epochs=3, dataset_size=1000, batch_size=32)
    expected = 3 * math.ceil(1000 / 32)
    assert a.total_steps == expected, (a.total_steps, expected)
    b = ScheduleArgs(epochs=2, dataset_size=64, batch_size=32)
    assert b.total_steps == 2 * math.ceil(64 / 32) == 4, b.total_steps
    c = ScheduleArgs(epochs=1, dataset_size=65, batch_size=32)
    assert c.total_steps == math.ceil(65 / 32) == 3, c.total_steps


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass

@dataclass
class ScheduleArgs:
    epochs: int = 3
    dataset_size: int = 1000
    batch_size: int = 32
    total_steps: int = 0

    def __post_init__(self):
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')
        self.total_steps = self.epochs * ((self.dataset_size + self.batch_size - 1) // self.batch_size)

args = ScheduleArgs(epochs=3, dataset_size=1000, batch_size=32)
print(args.total_steps)
```
</details>